In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

### 문서 분할 설정

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_upstage import ChatUpstage, UpstageEmbeddings

print("==> 0. 문서 로드")
PDF_PATH = "../data/콘텐츠분쟁해결_사례.pdf"
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(f"  총 {len(documents)}페이지 로드 완료")

print("==> 1. 문서 분할")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,       # 법률 사례는 1200~1800자 권장
    chunk_overlap=300,     # 사례 맥락 보존을 위해 200~400자
    separators=[
        "【사건개요】",     # PDF 추출 텍스트에는 앞에 줄바꿈이 없어서 "\n" 제거
        "【쟁점사항】",
        "【처리경위】",
        "【처리결과】",
        "\n■", "\n\n", "\n", ".", " ", ""
    ],
)
chunks = text_splitter.split_documents(documents)
print(f"  분할 직후: {len(chunks)}개 청크")

# 목차 청크 제외 (점선 '····' 이 많은 청크)
chunks = [c for c in chunks if c.page_content.count("····") < 5]
print(f"  목차 제외 후: {len(chunks)}개 청크")
print(f"  평균 청크 길이: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f}자")

### 임베딩 모델 설정

In [ ]:
DB_PATH = "../db/faiss_db_content_dispute"

print("==> 2. 임베딩 모델 설정 + 벡터스토어 저장")
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(DB_PATH)
print(f"  FAISS 인덱스 저장 완료: {DB_PATH} ({len(chunks)}개 벡터)")

### 검색기, LLM 설정

In [ ]:
print("==> 3. 검색기 설정")
retriever = vectorstore.as_retriever(
    search_type="similarity",      # 다양성이 필요하면 "mmr"
    search_kwargs={"k": 5},        # 상위 5개 관련 사례 검색
)

print("==> 4. LLM 설정")
llm = ChatUpstage(
    model="solar-pro3",
    base_url="https://api.upstage.ai/v1",
    temperature=0.2,
)
print(llm.model_name)

### 프롬포트 작성 ( 법률 자문 ), LCEL 체인 생성

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 합침"""
    return "\n\n".join(doc.page_content for doc in docs)


prompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다.
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요
2. 관련 법령이나 조항이 있다면 명시하세요
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요
4. 실무적 해결방안을 단계별로 제시하세요
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요

전문 법률 조언:"""

prompt = PromptTemplate.from_template(prompt_template)

# LCEL 체인
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("LCEL RAG 체인 구축 완료")

### 테스트 질문 작성

In [ ]:
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?",
]


def rag_with_sources(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    response = llm.invoke(prompt.format(context=context, question=question))
    return {"answer": response.content, "source_documents": docs}


print("=" * 60)
print(" 콘텐츠분쟁해결 RAG 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n【테스트 {i}/{len(test_questions)}】")
    print(f"Q: {question}")
    result = rag_with_sources(question)
    print(f"A: {result['answer']}")

    docs = result["source_documents"]
    print(f"  참조: {len(docs)}개 문서")
    for j, doc in enumerate(docs[:3], 1):
        page = doc.metadata.get("page_label", "?")
        print(f"    {j}. P{page}: {doc.page_content[:80]}...")
    print("-" * 40)

In [ ]:
def classify_dispute_type(query):
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]

    query_lower = query.lower()

    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"


for q in test_questions:
    print(f"[{classify_dispute_type(q)}] {q[:40]}...")